# Identification accuracy: model, mix or measurement?

One notebook, end to end, **without any network call**: it reads the ClickHouse cache (`data/cache/clickhouse`), the public
extracts (`data/processed`) and the dbt marts (`dbt/fingerprint.duckdb`, read-only). Labels: **PUBLIC DATA** (real) and
**SIMULATED** (fictional platform and customers). Nothing here is Fingerprint data.

Run order (README): fetch -> ClickHouse cache -> simulate -> `dbt build` -> monitoring -> findings -> this notebook.

In [1]:
import json, sys
from pathlib import Path
import duckdb, numpy as np, pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import metrics
CH = ROOT / "data" / "cache" / "clickhouse"
con = duckdb.connect(str(ROOT / "dbt" / "fingerprint.duckdb"), read_only=True)
F = json.loads((ROOT / "data" / "processed" / "findings.json").read_text())
ch = lambda name: pd.read_parquet(CH / f"{name}.parquet")
pd.set_option("display.width", 160)

## 1. Real data (PUBLIC DATA): Fingerprint's accuracy method replayed in ClickHouse

ClickBench `hits`: ~100M anonymised web-analytics events of July 2013. Control = the cookie (`UserID`), test = the user of the
last earlier event with the same 16-attribute fingerprint. The SQL is in `clickhouse/analyses/`; here we read the cached results.

In [2]:
p = ch("01_profile_totals").T
p.columns = ["value"]
p

,value
events,99997497
users_exact,17630976
users_uniq_approx,17578964
sites,6506
first_day,2013-07-02
last_day,2013-07-31
days_present,17
events_without_userid,0
bot_share,0.332837
mobile_share,0.101727


In [3]:
# the coverage trap: 17 days, very uneven volumes -> rates only
ch("01_profile_daily")[["EventDate", "events", "users", "bot_share"]]

,EventDate,events,users,bot_share
0,2013-07-02,3704833,841790,0.359429
1,2013-07-03,9829176,1795997,0.367424
2,2013-07-05,4489531,808488,0.295141
3,2013-07-06,6813120,1136041,0.282968
4,2013-07-07,1126867,153761,0.279782
5,2013-07-08,2350577,303237,0.329939
6,2013-07-09,13465700,1679702,0.320805
7,2013-07-10,5808207,1051295,0.349085
8,2013-07-15,12621831,3291214,0.403367
9,2013-07-20,429260,195098,0.319636


In [4]:
def shares(df):
    s = df[["tp_n", "fp_n", "fn_n", "tn_n"]].sum()
    return (s / s.sum()).round(4)
pd.DataFrame({"event": shares(ch("04_reid_event_grain")), "visit": shares(ch("05_reid_visit_grain")), "user-day": shares(ch("05_reid_day_grain"))})

,event,visit,user-day
tp_n,0.5669,0.0504,0.0065
fp_n,0.3650,0.7773,0.7907
fn_n,0.0334,0.0164,0.0061
tn_n,0.0348,0.1559,0.1967


**Reading**: the same method gives very different numbers at event and visit grain: repeated page views are easy matches. An identification request is closer to a visit.

In [5]:
t = pd.concat([ch("06_reid_tolerant_a"), ch("06_reid_tolerant_b")]).groupby("variant")[["tp_n", "fp_n", "fn_n", "tn_n"]].sum()
(t.div(t.sum(axis=1), axis=0)).round(4).rename(index={0: "strict 18", 1: "exact 16", 2: "tolerant 14", 3: "loose 9", 4: "loosest 5"})

,tp_n,fp_n,fn_n,tn_n
variant,,,,
strict 18,0.0724,0.6246,0.0335,0.2695
exact 16,0.0504,0.7773,0.0164,0.1559
tolerant 14,0.0382,0.8595,0.0074,0.0949
loose 9,0.0222,0.9221,0.0045,0.0512
loosest 5,0.0146,0.9630,0.0006,0.0218


In [6]:
# cross-engine reconciliation: DuckDB (dbt int_hits_reid) vs ClickHouse (analysis 11)
con.sql("select count(*) as site_grain_rows, count(distinct counter_id) as sites, sum(total_abs_diff) as total_abs_diff from mart_hits_reid_check").df()

,site_grain_rows,sites,total_abs_diff
0,134,67,0.0


In [7]:
# second method for one number: the event-grain replay of site 3922 recomputed from the per-site ClickHouse output
e = ch("04_reid_event_grain").set_index("CounterID").loc[3922]
round(e.fp_n / e.events, 4), F["real"]["replay"]["sites"][0]

(np.float64(0.2506),
 {'site': 3922,
  'events': 8527070,
  'visits': 777125,
  'tp_event': 0.7075,
  'fp_event': 0.2506,
  'tp_visit': 0.0331,
  'fp_visit': 0.7241})

## 2. The simulated platform (SIMULATED): measurement vs hidden truth

Each event has a hidden true device. Deterministic events also carry the answer the platform would give without the
deterministic signal (`probabilistic_visitor_id`): this is Fingerprint's control / test set.

In [8]:
con.sql('''
select sum(control_events) control, sum(fp) * 1.0 / sum(control_events) measured_fp, sum(fn) * 1.0 / sum(control_events) measured_fn,
       sum(truth_fp) * 1.0 / sum(truth_tp + truth_fp + truth_fn + truth_tn) true_fp_all, sum(truth_fn) * 1.0 / sum(truth_tp + truth_fp + truth_fn + truth_tn) true_fn_all
from fct_accuracy_daily''').df()

,control,measured_fp,measured_fn,true_fp_all,true_fn_all
0,8693963.0,0.009831,0.02365,0.007059,0.014475


## 3. The 8 hidden events, one by one

### Event 1: Chrome 155 drift (real release date 2026-09-23)

In [9]:
con.sql('''
select case when event_date < date '2026-10-02' then 'drift' else 'after fix' end as period, browser_major,
       sum(control_events) n, round(sum(fn) * 1.0 / sum(control_events), 4) fn_rate
from fct_accuracy_daily where browser in ('Chrome', 'Chrome Mobile') and browser_major in (154, 155)
  and event_date between date '2026-09-23' and date '2026-10-15' group by 1, 2 order by 2, 1''').df()

,period,browser_major,n,fn_rate
0,after fix,154,263281.0,0.0235
1,drift,154,99778.0,0.0228
2,after fix,155,107034.0,0.0239
3,drift,155,1439.0,0.0883


### Event 2: iOS 27 false positives and an overconfident score (real release date 2026-09-14)

In [10]:
pd.DataFrame(F["sim"]["e2"]).drop(columns=["c02_step_up_share"], errors="ignore").T.head(8)

,answers,avg_confidence,true_fp_rate,implied_fp_rate,before,issue,after
segment,"Mobile Safari, probabilistic answers with an e...","Mobile Safari, probabilistic answers with an e...","Mobile Safari, probabilistic answers with an e...","Mobile Safari, probabilistic answers with an e...","Mobile Safari, probabilistic answers with an e...","Mobile Safari, probabilistic answers with an e...","Mobile Safari, probabilistic answers with an e..."
before_other,309457.0,0.9747,0.0273,0.0253,NaN,NaN,NaN
after_ios27,569060.0,0.9746,0.0302,0.0254,NaN,NaN,NaN
issue_ios27,52514.0,0.9683,0.0573,0.0317,NaN,NaN,NaN
issue_other,291257.0,0.9746,0.0273,0.0254,NaN,NaN,NaN
after_other,1358921.0,0.9745,0.0307,0.0255,NaN,NaN,NaN
measured_fp_ios27_issue,0.0487,0.0487,0.0487,0.0487,0.0487,0.0487,0.0487
measured_fp_ios27_after,0.0245,0.0245,0.0245,0.0245,0.0245,0.0245,0.0245


In [11]:
pd.DataFrame(F["sim"]["e2"]["c02_step_up_share"])

,before,issue,after
all,0.0194,0.0212,0.0193
ios27,NaN,0.1123,0.0720
iphone_other,0.0702,0.0712,0.0697


### Event 3: C07, one visitor id with hundreds of accounts = support agents

In [12]:
con.sql("select scope, visitors, accounts_per_visitor_mean, accounts_per_visitor_p99, accounts_per_visitor_max from mart_linked_id_metrics where customer_id = 'C07' order by scope").df()

,scope,visitors,accounts_per_visitor_mean,accounts_per_visitor_p99,accounts_per_visitor_max
0,all,179804,1.917254,2.0,3712
1,excl_bad_bots,170585,1.891608,2.0,3712
2,excl_internal_and_bots,170545,1.061532,2.0,12


In [13]:
# the inference rule checked against the hidden truth (agents are known only because the data is simulated)
F["sim"]["e3"]["internal_rule"]

{'flagged': 40,
 'true_agents_flagged': 40,
 'non_agents_flagged': 0,
 'agents_missed': 0,
 'precision': 1.0,
 'recall': 1.0}

### Event 4: C03 reconciliation bridge (blocked requests, local time)

In [14]:
pd.DataFrame(F["sim"]["e4"]["bridge"])

,step,value,subtotal
0,"Customer export: sessions, October 2026 (local...",446141,NaN
1,minus sessions with no visitor id (identificat...,-39844,NaN
2,minus duplicate sessions in the export (page r...,-658,NaN
3,= our identification events on the same LOCAL ...,405639,True
4,minus / plus the time-zone shift (local days v...,689,NaN
5,"= billed identification events, October 2026 (...",406328,True


In [15]:
# second method: rebuild the bridge's first and last numbers from the raw export and from staging
exp = pd.read_csv(ROOT / "data" / "sample" / "C03_sessions_export.csv", parse_dates=["local_date"])
their = exp[exp.local_date.between("2026-10-01", "2026-10-31")].sessions.sum()
ours = con.sql("select count(*) from stg_identification_events where customer_id = 'C03' and event_date between date '2026-10-01' and date '2026-10-31'").fetchone()[0]
their, ours, round(their / ours - 1, 4)

(np.int64(446141), 406328, np.float64(0.098))

### Event 5: duplicates and a late partition, caught by dbt and absorbed by the incremental model

In [16]:
pd.DataFrame(F["sim"]["e5"]["incremental_check"]).T

,incremental_after_last_run,full_build,naive_pipeline
2026-10-25,197193,197193,197193
2026-10-26,239661,239661,239661
2026-10-27,240233,240233,245211
2026-10-28,233811,233811,0
2026-10-29,233983,233983,233983
2026-10-30,226990,226990,226990
2026-10-31,199146,199146,199146


### Event 6: robots in the P01 POC

In [17]:
{k: v for k, v in F["sim"]["e6"].items()}

{'fn_humans_outside': 0.0184,
 'fn_true_humans_wave': 0.0162,
 'fn_true_humans_outside': 0.0179,
 'bot_share_wave': 0.4082,
 'bot_share_outside': 0.0245,
 'fn_all_traffic_wave': 0.1881,
 'fn_humans_wave': 0.0412,
 'fn_bots_wave': 0.6025,
 'visitors_per_account_wave_all': 2.53,
 'visitors_per_account_wave_excl_bots': 1.52,
 'wave': ['2026-11-05', '2026-11-12']}

### Event 7: C09 private-browsing mix and the blind spot of the control set

In [18]:
pd.DataFrame(F["sim"]["e7"]["by_segment"]).T

,events,true_error_rate,true_fn_rate
after_incognito,103276.0,0.0881,0.0726
before_normal,545506.0,0.0137,0.0092
after_normal,410716.0,0.0131,0.0090
before_incognito,43542.0,0.0864,0.0712


**Reading**: within each segment the true error rate does not move; the total moves because the incognito share grows.
The measured rate (control set) cannot see incognito traffic, so it stays flat: optimistic.

### Event 8: the FALSE SIGNAL of the week of 2026-10-19

In [19]:
con.sql("select week_start, round(fp_rate_crude, 5) crude, round(fp_rate_standardized, 5) constant_mix, round(fp_rate_stable_cohort, 5) stable_cohort from mart_mix_standardized").df()

,week_start,crude,constant_mix,stable_cohort
0,2026-08-24,0.00850,0.00851,0.00850
1,2026-08-31,0.00844,0.00843,0.00844
2,2026-09-07,0.00857,0.00857,0.00857
3,2026-09-14,0.00850,0.00849,0.00850
4,2026-09-21,0.00871,0.00871,0.00871
5,2026-09-28,0.00880,0.00882,0.00880
6,2026-10-05,0.00866,0.00865,0.00866
7,2026-10-12,0.00837,0.00836,0.00836
8,2026-10-19,0.01086,0.00839,0.00844
9,2026-10-26,0.01124,0.00840,0.00844


In [20]:
# second method for the standardized rate: recompute it in pandas from the daily facts with metrics.direct_standardize
d = con.sql("select date_trunc('week', event_date) wk, browser, device_type, sum(control_events) n, sum(fp) fp from fct_accuracy_daily group by all").df()
ref = d[d.wk < "2026-10-19"].groupby(["browser", "device_type"]).n.sum()
w = (ref / ref.sum()).rename("w")
def std(week):
    g = d[d.wk == week].set_index(["browser", "device_type"]).join(w, how="inner")
    return metrics.direct_standardize(g.fp.values, g.n.values, g.w.values)
round(std(pd.Timestamp("2026-10-12")), 5), round(std(pd.Timestamp("2026-10-19")), 5)

(0.00836, 0.00839)

In [21]:
pd.DataFrame(F["sim"]["e8"]["customers_week_over_week"])

,customer_id,fp_a,fp_b,p_value
0,C01,0.0082,0.0082,0.9918
1,C02,0.0078,0.0077,0.8535
2,C03,0.0096,0.0099,0.7429
3,C04,0.0093,0.0096,0.6468
4,C05,0.0093,0.0087,0.4300
5,C06,0.0077,0.0094,0.0669
6,C07,0.0072,0.0072,0.9586
7,C08,0.0082,0.0082,0.9860
8,C09,0.0083,0.0083,0.9645
9,C10,0.0062,0.0071,0.3847


No existing customer changed significantly and the constant-mix line is flat: **mix, not model. No incident.**

## 4. Statistics

In [22]:
pd.DataFrame(F["sim"]["stats"]["design_effects"])

,customer_id,metric,from,events_per_visitor,max_events_one_visitor,rate,design_effect,width_ratio
0,C04,FP,2026-10-26,2.61,119,0.00929,1.07,1.05
1,C04,FN,2026-08-24,5.78,263,0.02367,1.29,1.12
2,C07,FN,2026-08-24,6.90,3623,0.02323,1.61,1.26
3,C12,FP,2026-10-19,4.41,245,0.01899,1.22,1.08


In [23]:
c4 = F["sim"]["stats"]["c04"]
print(f"C04 FP {c4['fp_rate']:.3%}; to detect +0.2 pts: {c4['required_n_per_period']:,} events per period = {c4['required_days_per_period']} days")
print("check with metrics.required_n:", round(metrics.required_n(c4["fp_rate"], 0.002, phi=c4["design_effect"])))

C04 FP 0.929%; to detect +0.2 pts: 38,714 events per period = 3.5 days
check with metrics.required_n: 38648


In [24]:
pd.DataFrame(F["sim"]["stats"]["threshold_tradeoff_c02"])

,threshold,step_up_share,fp_caught_share,precision_of_step_up
0,0.90,0.0040,0.1099,0.1649
1,0.92,0.0071,0.1629,0.1393
2,0.94,0.0137,0.2424,0.1077
3,0.95,0.0198,0.3022,0.0924
4,0.96,0.0301,0.3850,0.0776
5,0.97,0.0481,0.4875,0.0615
6,0.98,0.0834,0.6414,0.0467
7,0.99,0.1623,0.8402,0.0314


## 5. Monitoring and back-test

In [25]:
bt = F["monitoring"]
{k: bt[k] for k in ["event_1", "event_2", "event_5", "event_6", "event_8", "segment_alerts", "false_alerts_per_week_all_monitors"]}

{'event_1': {'detected': True,
  'first_alert': '2026-09-26',
  'delay_days': 3,
  'alerting_series': 3},
 'event_2': {'detected': True,
  'first_alert': '2026-09-15',
  'delay_days': 1,
  'alerting_series': 5},
 'event_5': {'detected': True,
  'alerts': 12,
  'robot_wave_volume_alerts': 0,
  'false_volume_alerts': 1},
 'event_6': {'detected': True,
  'first_alert': '2026-11-05',
  'false_bot_alerts': 0},
 'event_8': {'naive_global_signal_days_after_start': 35,
  'naive_global_signal_days_before_start': 1,
  'standardized_signal_days_after_start': 0,
  'segment_alerts_after_start_excluding_events_1_2': 2},
 'segment_alerts': {'alert_days_total': 46,
  'true_alert_days': 44,
  'precision': 0.957,
  'false_alert_days_per_week': 0.15,
  'recall_events_1_2': 1.0},
 'false_alerts_per_week_all_monitors': 0.23}

In [26]:
pd.read_csv(ROOT / "data" / "processed" / "monitoring_alerts.csv").head(15)

,level,metric,customer_id,browser,browser_major,event_date,n,rate,p0,ref_source,z,p,true_alert,matched_event
0,customer_segment,fp,C02,Mobile Safari,27,2026-09-27,206.0,0.058252,0.025535,rest of family,2.650659,2.016526e-05,True,2.0
1,customer_segment,fp,C04,Mobile Safari,27,2026-09-27,241.0,0.058091,0.025749,rest of family,2.822668,2.959172e-05,True,2.0
2,customer_segment,fp,C04,Mobile Safari,27,2026-10-06,316.0,0.066456,0.027699,rest of family,3.738086,4.920638e-05,True,2.0
3,customer_segment,fp,C08,Mobile Safari,27,2026-09-22,119.0,0.084034,0.024055,rest of family,3.802383,2.621916e-05,True,2.0
4,customer_segment,fp,C09,Mobile Safari,27,2026-10-06,143.0,0.062937,0.027612,rest of family,2.295554,3.512105e-05,True,2.0
5,customer_segment,fn,C01,Firefox,158,2026-10-19,346.0,0.063584,0.028378,rest of family,3.812722,1.717733e-05,False,NaN
6,customer_segment,fn,C10,Chrome,155,2026-10-01,23.0,0.260870,0.022243,rest of family,NaN,1.588766e-05,True,1.0
7,customer_segment,fn,P01,Chrome,146,2026-11-09,151.0,0.211921,0.061256,rest of family,7.463917,6.901619e-11,True,6.0
8,customer_segment,fn,P01,Chrome,146,2026-11-10,121.0,0.165289,0.074538,rest of family,3.674431,4.116601e-11,True,6.0
9,customer_segment,fn,P01,Chrome,146,2026-11-11,124.0,0.217742,0.080830,rest of family,5.407314,2.432069e-08,True,6.0


Same detector on a real series (Wikimedia unique devices): alerts are listed, no cause claimed.

In [27]:
F["public"]["unique_devices"]["alert_days"]

['2025-07-25', '2025-07-26', '2025-07-27']

## 6. Self-service tools (read-only, aggregates only)

In [28]:
sys.path.insert(0, str(ROOT / "tools"))
import accuracy_core as core
print(core.accuracy_summary("C07", "2026-10-26", "2026-11-22")["summary"])
print(core.is_change_significant("C04", "fp", "2026-10-05", "2026-10-18", "2026-10-19", "2026-11-01")["summary"])
print(core.customer_health("P01")["summary"])

C07, 2026-10-26 to 2026-11-22: measured false-positive rate 0.75% (95% CI 0.72%-0.78%), false-negative rate 2.30%, on 311,643 control events.
C04: the false-positive rate went from 0.94% to 0.93% (-2%). With 149,737 and 152,262 control events and the usual day-to-day noise (over-dispersion 1.3), this is within normal noise (p = 0.678).
P01 is GREEN for the week of 2026-11-16: no accuracy change, robot or incognito alert, no urgent request.


*Independent case study based on public data · not affiliated with or endorsed by Fingerprint*